# 24. Text Vectorization

## Overview
Machine learning algorithms generally require numerical input. Text vectorization converts documents into numerical feature representations that a model can process.

This notebook introduces the vectorization workflow and compares common representations, including count-based features and TF-IDF.

## 1. Import libraries

The examples use scikit-learn's text feature extraction tools.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

## 2. Create a small document collection

Each string is a document. The dataset is intentionally small so that the resulting feature matrices are easy to inspect.

In [2]:
documents = [
    "machine learning improves text analysis",
    "text analysis uses machine learning",
    "natural language processing analyzes text",
    "machine learning models learn patterns",
]

for index, document in enumerate(documents, start=1):
    print(f"Document {index}: {document}")

Document 1: machine learning improves text analysis
Document 2: text analysis uses machine learning
Document 3: natural language processing analyzes text
Document 4: machine learning models learn patterns


## 3. Convert documents to a document-term matrix

`CountVectorizer` builds a vocabulary and counts each vocabulary term in each document.

In [3]:
count_vectorizer = CountVectorizer()
count_matrix = count_vectorizer.fit_transform(documents)

count_df = pd.DataFrame(
    count_matrix.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=[f"doc_{i}" for i in range(1, len(documents) + 1)],
)

display(count_df)

,analysis,analyzes,improves,language,learn,learning,machine,models,natural,patterns,processing,text,uses
doc_1,1,0,1,0,0,1,1,0,0,0,0,1,0
doc_2,1,0,0,0,0,1,1,0,0,0,0,1,1
doc_3,0,1,0,1,0,0,0,0,1,0,1,1,0
doc_4,0,0,0,0,1,1,1,1,0,1,0,0,0


## 4. Convert documents to TF-IDF features

TF-IDF downweights terms that occur in many documents and emphasizes terms that are more distinctive within a document collection.

In [4]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"doc_{i}" for i in range(1, len(documents) + 1)],
)

display(tfidf_df.round(3))

,analysis,analyzes,improves,language,learn,learning,machine,models,natural,patterns,processing,text,uses
doc_1,0.468,0.000,0.593,0.000,0.000,0.378,0.378,0.000,0.000,0.000,0.000,0.378,0.000
doc_2,0.468,0.000,0.000,0.000,0.000,0.378,0.378,0.000,0.000,0.000,0.000,0.378,0.593
doc_3,0.000,0.476,0.000,0.476,0.000,0.000,0.000,0.000,0.476,0.000,0.476,0.304,0.000
doc_4,0.000,0.000,0.000,0.000,0.512,0.327,0.327,0.512,0.000,0.512,0.000,0.000,0.000


## 5. Compare matrix shapes and sparsity

Text matrices are usually sparse: most documents contain only a small fraction of the full vocabulary.

In [5]:
def matrix_summary(matrix, name):
    total = matrix.shape[0] * matrix.shape[1]
    nonzero = matrix.nnz
    sparsity = 1 - nonzero / total if total else 0
    print(f"{name}: shape={matrix.shape}, nonzero={nonzero}, sparsity={sparsity:.2%}")

matrix_summary(count_matrix, "Count matrix")
matrix_summary(tfidf_matrix, "TF-IDF matrix")

Count matrix: shape=(4, 13), nonzero=20, sparsity=61.54%
TF-IDF matrix: shape=(4, 13), nonzero=20, sparsity=61.54%


## 6. Inspect a new document

Use `transform`, not `fit_transform`, for new documents after a vectorizer has been fitted. This preserves the vocabulary learned from the training collection.

In [6]:
new_documents = ["machine learning analyzes language"]
new_count_matrix = count_vectorizer.transform(new_documents)
new_tfidf_matrix = tfidf_vectorizer.transform(new_documents)

print("Count features:", new_count_matrix.toarray())
print("TF-IDF features:", new_tfidf_matrix.toarray())

Count features: [[0 1 0 1 0 1 1 0 0 0 0 0 0]]
TF-IDF features: [[0.         0.59603894 0.         0.59603894 0.         0.38044393
  0.38044393 0.         0.         0.         0.         0.
  0.        ]]


## Key takeaways

- Vectorization converts text into numeric features.
- Count-based features represent term frequency.
- TF-IDF weights terms by their relative importance within a corpus.
- Fit vectorizers on training data only, then transform validation and test data to avoid data leakage.